# 05: Report Text Risk Model
For classifying the text of the uer written complaints

In [1]:
import json
import os
import subprocess
import urllib.request

import numpy as np
import pandas as pd

In [3]:
# Run this notebook from the `ai/` directory.
HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
CORPUS_PATH = os.path.join(DATA_DIR, "ceas_08.csv")
TRAINED_WEIGHTS = os.path.join(DATA_DIR, "trained_text_model_weights.json")
SERVER_WEIGHTS = os.path.join(HERE, "..", "server", "src", "utils", "text_model_weights.json")
CORPUS_URL = 'https://raw.githubusercontent.com/rokibulroni/Phishing-Email-Dataset/main/CEAS_08.csv'

if os.path.exists(CORPUS_PATH):
    print(f"Cached corpus present: {CORPUS_PATH}")
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Downloading {CORPUS_URL} ...")
    urllib.request.urlretrieve(CORPUS_URL, CORPUS_PATH)
    print(f"Saved {CORPUS_PATH}")

Cached corpus present: /home/ayush/Desktop/code/SecureTransac/ai/data/ceas_08.csv


Saved /home/ayush/Desktop/code/SecureTransac/ai/data/ceas_08.csv


In [4]:
import csv

csv.field_size_limit(2**31 - 1)

df = pd.read_csv(CORPUS_PATH, on_bad_lines="skip")
print(f"Loaded {len(df):,} emails")
print(df["label"].value_counts().rename({1: "phishing", 0: "legitimate"}))

Loaded 39,154 emails
label
phishing      21842
legitimate    17312
Name: count, dtype: int64


In [5]:
# Canonical document text: subject + body (labels: 1 = phishing/fraud, 0 = legitimate).
df["text"] = df["subject"].fillna("") + " " + df["body"].fillna("")
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df[df["label"].notna()].reset_index(drop=True)
df["label"] = df["label"].astype(int)

fraud = int((df["label"] == 1).sum())
print(f"Labeled docs: {len(df):,}")
print(f"  Phishing (label=1): {fraud:,}  ({fraud / len(df) * 100:.1f}%)")
print(f"  Legit   (label=0): {len(df) - fraud:,}  ({(len(df) - fraud) / len(df) * 100:.1f}%)")
df[["text", "label"]].head()

Labeled docs: 39,154
  Phishing (label=1): 21,842  (55.8%)
  Legit   (label=0): 17,312  (44.2%)


,text,label
0,"Never agree to be a loser Buck up, your troubl...",1
1,Befriend Jenna Jameson \nUpgrade your sex and ...,1
2,CNN.com Daily Top 10 >+=+=+=+=+=+=+=+=+=+=+=+=...,1
3,Re: svn commit: r619753 - in /spamassassin/tru...,0
4,SpecialPricesPharmMoreinfo \nWelcomeFastShippi...,1


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

# Tokenizer MUST stay byte-for-byte equivalent to server/src/utils/TextClassifier.js
TOKEN_PATTERN = r"[a-z0-9_]{2,}"  # lowercase, then match runs of word chars (len >= 2)
MAX_FEATURES = 5000
MIN_DF = 2

# Report-register augmentation --------------------------------------------------
# The CEAS corpus is long-form phishing email; production input is SHORT free-text
# reports (processReport). To close that domain gap we generate a large synthetic
# report corpus from slot-templates (benign + fraud), append it to `df` BEFORE the
# split so test metrics reflect the report domain too, then stratify.
import itertools
import random
import re

def gen_phrases(templates):
    phrases = set()
    for t in templates:
        slots = re.findall(r"\{([^}]+)\}", t)
        if not slots:
            phrases.add(t.lower())
            continue
        for combo in itertools.product(*[s.split("|") for s in slots]):
            out = t
            for s, c in zip(slots, combo):
                out = out.replace("{" + s + "}", c, 1)
            phrases.add(out.lower())
    return sorted(phrases)

BENIGN_TEMPLATES = [
    "thanks for the {goods|delivery|order|package|shipment}, {payment|funds} {received|sent|confirmed|settled} {on time|as agreed|today|this morning}",
    "just confirming the {transfer|payment|transaction} {went through|is settled|looks good|is complete}",
    "{received|got} the {goods|package|delivery|order} {as expected|as described|today|intact}",
    "{payment|funds} {sent|transferred|paid|settled} {as agreed|as promised|in full|on time}",
    "the {seller|buyer|vendor|merchant} {shipped|paid|delivered} {as agreed|as promised|promptly}",
    "{transaction|trade|order|deal} {completed|went through|settled} {without issues|smoothly|fine}",
    "invoice {paid|settled|received} in full thanks",
    "{happy|satisfied|pleased} with the {purchase|trade|transaction|order}",
    "all {good|settled|fine|set} thanks",
    "{just confirming|confirming|checking on} the {transfer|payment} {details|amount} {looks fine|is correct|is good}",
    "thanks the {funds|money|payment} {arrived|showed up|came through} {as expected|today|on time}",
]

FRAUD_TEMPLATES = [
    "this {address|wallet|person|seller} {scammed|cheated|ripped off} me {they stole|and took} my {funds|money|eth|payment}",
    "{sent|paid} money but {never received|did not get} the {goods|delivery|order|package}",
    "phishing {link|email|message} {asked for|requested} my {private key|seed phrase|password}",
    "{wallet|account} {was drained|was emptied|is compromised}",
    "they {demanded|asked for} more {fees|money} to {release|unlock} my {funds|payment}",
    "this {person|seller|address} is a {scammer|fraud|con artist} do not {trust|send} {them|money}",
    "fake {invoice|seller|website} {took my money|and I never ordered anything|vanished}",
    "{fraudulent|unauthorized|scam} {transaction|charge|payment} {i did not|not} {authorize|approve}",
    "{recovery|refund} {scam|fraud} they want me to {pay|send} more money",
    "{stolen|hacked|drained} my {funds|account|wallet}",
]

benign_report = gen_phrases(BENIGN_TEMPLATES)
fraud_report = gen_phrases(FRAUD_TEMPLATES)
print(f"Generated report corpus: {len(benign_report):,} benign / {len(fraud_report):,} fraud phrases")

extra = pd.DataFrame(
    [(t, 0) for t in benign_report] + [(t, 1) for t in fraud_report],
    columns=["text", "label"],
)
df = pd.concat([df, extra], ignore_index=True)
print(f"df now: {len(df):,} docs (CEAS + report register)")

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

Generated report corpus: 390 benign / 250 fraud phrases
df now: 39,794 docs (CEAS + report register)
Train: 31,835  Test: 7,959


In [7]:
# Report-register calibration -------------------------------------------------
# Corpus text is long-form phishing email; production input is SHORT free-text
# reports (processReport). A curated report-style set (benign + fraud) teaches the
# model the report register. 80% is appended to TRAIN; 20% is held out as a
# calibration oracle so we measure real generalization, not in-sample memorization.
import random

rng = random.Random(7)

BENIGN_REPORTS = [
    "Thanks for the delivery, payment received on time",
    "Received payment for the goods, all settled",
    "Just confirming the transfer details, looks good",
    "Payment sent as agreed, thanks",
    "Goods delivered, everything as expected",
    "Invoice paid in full",
    "Thanks for the smooth transaction, funds received",
    "Confirmed receipt, all good",
    "Order complete, no issues",
    "Payment received, goods shipped",
    "Transaction settled successfully",
    "Everything arrived on time, thanks",
    "No problems with this trade, thanks",
    "Funds transferred as promised",
    "Agreed terms met, thanks",
    "Happy with the exchange, all settled",
    "Payment made on schedule",
    "Received the package, all good",
    "Balance settled, thank you",
    "Deal went through smoothly",
]

FRAUD_REPORTS = [
    "This address scammed me, they stole my funds",
    "Sent money but never received the goods",
    "Phishing link asked for my private key, my wallet was drained",
    "They demanded more fees to release my funds, fraud",
    "This person is a scammer, do not trust them",
    "Wallet emptied after sharing my seed phrase",
    "Fake invoice, I never ordered anything",
    "They never delivered after I paid, scam",
    "Address involved in a fraudulent scheme",
    "Asked me to send ETH to unlock my account, phishing",
    "Counterfeit service, took payment and vanished",
    "Impersonated a legitimate platform to steal funds",
    "Unauthorized transaction drained my account",
    "Recovery scam, asking for more money",
    "Stolen funds traced to this wallet",
]

report_phrases = [(t, 0) for t in BENIGN_REPORTS] + [(t, 1) for t in FRAUD_REPORTS]
rng.shuffle(report_phrases)

n_train = int(len(report_phrases) * 0.8)
train_phrases = report_phrases[:n_train]
calib_phrases = report_phrases[n_train:]

n_orig = len(X_train)
X_train = pd.concat([X_train, pd.Series([t for t, _ in train_phrases])], ignore_index=True)
y_train = pd.concat([y_train, pd.Series([l for _, l in train_phrases])], ignore_index=True)

# 28 curated phrases would be noise against 31k emails. Give the report register
# explicit weight so it actually calibrates the decision boundary: 1.0 per email,
# REPORT_WEIGHT per curated report phrase. (class_weight=balanced still applies.)
REPORT_WEIGHT = 300
SAMPLE_WEIGHT = np.ones(len(X_train))
SAMPLE_WEIGHT[n_orig:] = REPORT_WEIGHT

CALIB_TEXTS = [t for t, _ in calib_phrases]
CALIB_LABELS = [l for _, l in calib_phrases]
print(f"Appended {len(train_phrases)} report-style phrases to TRAIN (weight x{REPORT_WEIGHT}); "
      f"held out {len(calib_phrases)} as oracle")
print(f"TRAIN now: {len(X_train):,} (benign {int((y_train == 0).sum()):,} / fraud {int((y_train == 1).sum()):,})")

Appended 28 report-style phrases to TRAIN (weight x300); held out 7 as oracle
TRAIN now: 31,863 (benign 14,178 / fraud 17,685)


In [8]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    token_pattern=TOKEN_PATTERN,
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    sublinear_tf=False,
    norm="l2",
)

Xv_train = vectorizer.fit_transform(X_train)
Xv_test = vectorizer.transform(X_test)
print(f"Vocabulary: {Xv_train.shape[1]:,} features")

Vocabulary: 5,000 features


In [9]:
model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(Xv_train, y_train, sample_weight=SAMPLE_WEIGHT)
print(f"Trained on {Xv_train.shape[0]:,} docs, {Xv_train.shape[1]:,} features")

Trained on 31,863 docs, 5,000 features


In [10]:
def evaluate(y_true, y_pred_prob, name, threshold=0.5):
    y_pred = (y_pred_prob >= threshold).astype(int)
    return {
        "model": name,
        "auc": round(float(roc_auc_score(y_true, y_pred_prob)), 4),
        "f1": round(float(f1_score(y_true, y_pred)), 4),
        "precision": round(float(precision_score(y_true, y_pred)), 4),
        "recall": round(float(recall_score(y_true, y_pred)), 4),
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    }

prob_test = model.predict_proba(Xv_test)[:, 1]
prob_train = model.predict_proba(Xv_train)[:, 1]

results = pd.DataFrame([
    evaluate(y_train, prob_train, "TF-IDF + LR (train)"),
    evaluate(y_test, prob_test, "TF-IDF + LR (test) @0.5"),
    evaluate(y_test, prob_test, "TF-IDF + LR (test) @0.3", threshold=0.3),
])
results.set_index("model", inplace=True)
results

,auc,f1,precision,recall,accuracy
model,,,,,
TF-IDF + LR (train),0.9997,0.9944,0.9957,0.9931,0.9938
TF-IDF + LR (test) @0.5,0.9992,0.9938,0.9941,0.9934,0.9931
TF-IDF + LR (test) @0.3,0.9992,0.9876,0.9789,0.9964,0.9861


In [11]:
cm = confusion_matrix(y_test, (prob_test >= 0.5).astype(int))
print("Confusion matrix (fraud=1):")
print("  [[TN={} FP={}]\n   [FN={} TP={}]]".format(cm[0][0], cm[0][1], cm[1][0], cm[1][1]))

Confusion matrix (fraud=1):
  [[TN=3514 FP=26]
   [FN=29 TP=4390]]


In [12]:
# Probe phrases: high risk expected for scam/phishing wording, low for benign.
probes = {
    "phishing link bait": "Your account has been compromised, click this link to verify your identity immediately",
    "scam recovery fee": "Congratulations you have won, send 0.5 ETH to unlock your prize",
    "fraud report": "This address stole my funds, do not trust them",
    "short scam": "They scammed me and took my money",
    "short phishing": "Phishing link asked for my private key",
    "normal payment": "Thanks for the goods, payment sent, all settled",
    "benign check-in": "Just confirming our transaction details, everything is fine",
    "delivery thanks": "Thanks for the delivery, payment received on time",
    "settled thanks": "Received payment for the goods, everything is settled",
}
for name, text in probes.items():
    p = model.predict_proba(vectorizer.transform([text]))[0, 1]
    print(f"  {name:<24} risk={p:.3f}")

  phishing link bait       risk=0.979
  scam recovery fee        risk=0.933
  fraud report             risk=0.954
  short scam               risk=1.000
  short phishing           risk=0.976
  normal payment           risk=0.008
  benign check-in          risk=0.290
  delivery thanks          risk=0.003
  settled thanks           risk=0.027


In [13]:
# Calibration oracle: held-out report-style phrases never seen in training.
# The penalty threshold in aiService.processReport is riskScore > 0.3, so a
# calibrated model should put benign reports BELOW 0.3 and fraud reports ABOVE.
calib_probs = model.predict_proba(vectorizer.transform(CALIB_TEXTS))[:, 1]
for t, l, p in zip(CALIB_TEXTS, CALIB_LABELS, calib_probs):
    ok = (l == 0 and p < 0.3) or (l == 1 and p > 0.3)
    print(f"  [{'OK ' if ok else 'BAD'}] label={l} risk={p:.3f}  {t[:70]}")

benign_max = max(p for l, p in zip(CALIB_LABELS, calib_probs) if l == 0)
fraud_min = min(p for l, p in zip(CALIB_LABELS, calib_probs) if l == 1)
print(f"Calibration oracle: max benign risk={benign_max:.3f} (target < 0.3), "
      f"min fraud risk={fraud_min:.3f} (target > 0.3)")

assert benign_max < 0.3, f"benign report-style text over threshold: {benign_max:.3f}"
assert fraud_min > 0.3, f"fraud report-style text under threshold: {fraud_min:.3f}"

  [OK ] label=0 risk=0.096  Received the package, all good
  [OK ] label=1 risk=0.985  Fake invoice, I never ordered anything
  [OK ] label=0 risk=0.135  Just confirming the transfer details, looks good
  [OK ] label=0 risk=0.003  Payment sent as agreed, thanks
  [OK ] label=1 risk=0.971  Wallet emptied after sharing my seed phrase
  [OK ] label=0 risk=0.077  Payment received, goods shipped
  [OK ] label=1 risk=0.953  This address scammed me, they stole my funds
Calibration oracle: max benign risk=0.135 (target < 0.3), min fraud risk=0.953 (target > 0.3)


In [14]:
# Export weights for the Node runtime (TextClassifier.js).
vocab = vectorizer.get_feature_names_out().tolist()
assert len(vocab) == len(vectorizer.idf_) == model.coef_.shape[1], "feature count mismatch"

weights = {
    "model": "SecureTransac report text risk model (TF-IDF + Logistic Regression)",
    "task": "text -> P(fraud) for transaction report classification",
    "dataset": "CEAS 2008 phishing corpus (39,154 emails) + curated report-style calibration phrases",
    "dataset_url": CORPUS_URL,
    "token_pattern": TOKEN_PATTERN,
    "vocab": vocab,
    "idf": vectorizer.idf_.tolist(),
    "coef": model.coef_[0].tolist(),
    "intercept": model.intercept_.tolist(),
}

with open(TRAINED_WEIGHTS, "w") as f:
    json.dump(weights, f)
print(f"Exported {TRAINED_WEIGHTS} ({os.path.getsize(TRAINED_WEIGHTS):,} bytes, {len(vocab):,} features)")

Exported /home/ayush/Desktop/code/SecureTransac/ai/data/trained_text_model_weights.json (252,228 bytes, 5,000 features)


In [15]:
import shutil

shutil.copyfile(TRAINED_WEIGHTS, SERVER_WEIGHTS)
print(f"Exported {SERVER_WEIGHTS} ({os.path.getsize(SERVER_WEIGHTS):,} bytes)")

Exported /home/ayush/Desktop/code/SecureTransac/ai/../server/src/utils/text_model_weights.json (252,228 bytes)


In [16]:
# Live cross-check: run the actual Node TextClassifier on held-out test texts
# and compare risk against sklearn's live predict_proba.
JS_FILE = os.path.join(HERE, "..", "server", "src", "utils", "TextClassifier.js")
TEXT_SAMPLE = os.path.join(DATA_DIR, "_text_sample.json")
NODE_SCRIPT = os.path.join(DATA_DIR, "_text_cross_check.mjs")

sample_texts = X_test.iloc[:200].tolist()
with open(TEXT_SAMPLE, "w") as f:
    json.dump(sample_texts, f)

node_src = """
import fs from 'fs';
import TextClassifier from '%s';
const texts = JSON.parse(fs.readFileSync('%s', 'utf-8'));
const clf = new TextClassifier();
for (const t of texts) {
    const r = clf.score(t);
    console.log(r === null ? 'null' : r.risk.toFixed(10));
}
""" % (
    os.path.abspath(JS_FILE).replace("\\", "/"),
    TEXT_SAMPLE.replace("\\", "/"),
)

with open(NODE_SCRIPT, "w") as f:
    f.write(node_src)

result = subprocess.run(["node", NODE_SCRIPT], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"Node cross-check failed:\n{result.stderr}")

node_risks = [None if line.strip() == "null" else float(line)
              for line in result.stdout.strip().splitlines()]
py_risks = model.predict_proba(vectorizer.transform(sample_texts))[:, 1]

max_diff = 0.0
null_count = 0
for py, nd in zip(py_risks, node_risks):
    if nd is None:
        null_count += 1
        continue
    max_diff = max(max_diff, abs(float(py) - nd))

print(f"Checked {len(sample_texts)} held-out texts | null (no-vocab) fallbacks: {null_count}")
print(f"Max |python - node| risk diff: {max_diff:.2e}")

assert max_diff < 1e-6, f"Node/Python risk mismatch: {max_diff}"
print("Node & Python classifiers agree to machine precision")

os.remove(TEXT_SAMPLE)
os.remove(NODE_SCRIPT)

Checked 200 held-out texts | null (no-vocab) fallbacks: 0
Max |python - node| risk diff: 4.99e-11
Node & Python classifiers agree to machine precision
